In [ ]:
import os
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '.99'

from euclidean_fast_attention.utils.npz_trainer import NpzTrainer
from euclidean_fast_attention.model import EnergyModel
from pathlib import Path
from orbax import checkpoint

import jax
import jax.numpy as jnp
import numpy as np
import optax

D2N = {10: 39, 15: 132, 20: 314, 25: 613, 30: 1060}
jax.local_devices()

2025-09-15 17:29:37.323892: W external/xla/xla/service/platform_util.cc:220] unable to create StreamExecutor for CUDA:0: : CUDA_ERROR_OUT_OF_MEMORY: out of memory


RuntimeError: Unable to initialize backend 'cuda': INTERNAL: no supported devices found for platform CUDA (you may need to uninstall the failing plugin package, or set JAX_PLATFORMS=cpu to skip this backend.)

In [ ]:
batch_size = 4

# IO.
save_folder = 'NaCl_toy_constant_density/'

# Data.
Ds = [10, 15, 20, 25, 30]

# Model.
T = 1
model = 'efa'
symmetry = 'exact'

# Training.
num_epochs = 100

for D in Ds:
    num_atoms = D2N[D]

    lambda_f = 10.0
    lambda_e = 1.0

    data_dir = Path('/home/thorben.frank/data/euclidean_fast_attention/').resolve() / f'NaCl_toy/vacuum/N{num_atoms}_D{D}.npz'
    data = np.load(data_dir)

    forces_weight = (lambda_f / data['forces'].var()).item()
    energy_weight = (lambda_e / data['energy'].var()).item()

    print(f'{energy_weight=} and {forces_weight=}')

    trainer = NpzTrainer(
        data_dir,
        num_train=1000,
        num_valid=200,
        max_num_nodes=batch_size*num_atoms+1,
        max_num_edges=batch_size*num_atoms*34+1,
        max_num_graphs=batch_size+1,
        num_epochs=num_epochs,
        energy_unit=1.,
        length_unit=1.,
        save_interval_steps=250,
        log_interval_steps=1_000_000,
        pbc_bool=False,
        use_wandb=False,
        forces_weight=forces_weight,
        energy_weight=energy_weight,
        subtract_energy_mean=False,
        rotation_augmentation_bool=True
    )
    
    base_model = EnergyModel(
        cutoff=5.0,
        num_features=32,
        num_layers=T,
        mp_max_degree=0,
        mp_block_post_mlp_bool=False,
        efa_block_post_mlp_bool=False,
        num_post_residual_mlps=1,
        era_activation_fn=lambda u: u,
        era_qk_num_features=128,
        era_v_num_features=8,
        era_use_in_iterations=list(range(T)) if model == 'efa' else [],
        era_max_degree=0,
        era_max_length=30.0,
        era_lebedev_num=194,
        era_max_frequency=4*np.pi,
        era_frequencies_trainable=False,
        pbc_bool=False
    )

    ckpt_dir = Path(f'{save_folder}/NaCl_toy_N{num_atoms}_D{D}_{model}_T{T}').resolve()
    trainer.run_training(
        ckpt_dir=ckpt_dir,
        model=base_model,
        optimizer=optax.adamw(1e-3),
    )

energy_weight=1.2416571735229809e-06 and forces_weight=0.5884617567062378


2025-09-05 13:24:39.156572: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2025-09-05 13:24:52.989505: W external/xla/xla/tsl/framework/bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 7.44GiB (rounded to 7988355072)requested by op 
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2025-09-05 13:24:52.989770: W external/xla/xla/tsl/framework/bfc_allocator.cc:512] **_____***__***__***************************************************************____________________
E0905 13:24:52.989790 3416499 pjrt_stream_executor_client.cc:2916] Execution of replica 0 failed: RESOURCE_

XlaRuntimeError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 7988355072 bytes.

: 

In [ ]:
model = 'efa'
batch_size = 4
T = 1
D = 30
freq_factor = 4

num_atoms = D2N[D]

data_dir = Path('/home/thorben.frank/data/euclidean_fast_attention/').resolve() / f'NaCl_toy/vacuum/N{num_atoms}_D{D}.npz'
data = np.load(data_dir)

trainer = NpzTrainer(
    data_dir,
    num_train=1000,
    num_valid=200,
    max_num_nodes=batch_size*num_atoms+1,
    max_num_edges=batch_size*num_atoms*35+1,
    max_num_graphs=batch_size+1,
    num_epochs=100,
    energy_unit=1.,
    length_unit=1.,
    save_interval_steps=250,
    log_interval_steps=1_000_000,
    pbc_bool=False,
    use_wandb=False,
    forces_weight=1.0,
    energy_weight=1.0,
    subtract_energy_mean=False
)

base_model = EnergyModel(
    cutoff=5.0,
    num_features=32,
    num_layers=T,
    mp_max_degree=0,
    mp_block_post_mlp_bool=False,
    efa_block_post_mlp_bool=False,
    num_post_residual_mlps=1,
    era_activation_fn=lambda u: u,
    era_qk_num_features=128,
    era_v_num_features=8,
    era_use_in_iterations=list(range(T)) if model == 'efa' else [],
    era_max_degree=0,
    era_max_length=30.0,
    era_lebedev_num=194,
    era_max_frequency=freq_factor*np.pi,
    era_frequencies_trainable=False,
    pbc_bool=False
)

ckpt_dir = Path(f'{save_folder}/NaCl_toy_N{num_atoms}_D{D}_{model}_T{T}').resolve()
loaded_mngr = checkpoint.CheckpointManager(
        ckpt_dir,
        {
            'params': checkpoint.PyTreeCheckpointer(),
            'opt_state': checkpoint.PyTreeCheckpointer(),
        },
        options=checkpoint.CheckpointManagerOptions(step_prefix='ckpt'),
    )
mgr_state = loaded_mngr.restore(loaded_mngr.latest_step())
params = mgr_state.get('params')

metrics, predictions = trainer.run_testing(params, base_model, num_test=200, collect_predictions=True)
energy_predictions, forces_predictions, energy_gt, forces_gt, graphs = predictions
energy_predictions_np = jnp.concatenate(energy_predictions).reshape(-1)
energy_gt_np = jnp.concatenate(energy_gt).reshape(-1)
forces_predictions_np = jnp.concatenate(forces_predictions)
forces_gt_np = jnp.concatenate(forces_gt)

save_filename = f'{save_folder}/NaCl_toy_N{num_atoms}_D{D}_{model}_T{T}'
np.savez(
    save_filename, 
    energy_predictions=energy_predictions_np,
    energy_gt=energy_gt_np,
    forces_predictions=forces_predictions_np,
    forces_gt=forces_gt_np,
)

2025-09-03 13:57:35.076642: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.


Stop testing after n=49 batches since 200 test structures have been reached.
Metrics are collected from 200 test structures.
